# BIOM5350 — In-Class Activity: Coding with AI II

This is the second of two in-class activities. Today you will work with an AI agent to answer one question:

> **Which genes are consistently differentially expressed in dilated cardiomyopathy (DCM) compared with non-failing hearts, across two independent studies (GSE5406 and GSE42955)?**

You will:

1. Write a **project instruction file** that tells the agent the facts and rules of this project.
2. Build an **analysis plan** with the agent, by having it interview you, and audit the plan before anything runs.
3. **Execute** the plan for the first study, checking each step against what you expected.
4. **Read** the code the agent wrote: have it explained, commented and restyled, and verify the results did not change.
5. Execute the plan for the second study.
6. **Combine** the two sets of results into a consensus gene list.
7. **Publish** the plan, the chat, the code and the results to GitHub, with a statement of how AI was used.

The agent will write almost all of the code today. Your job is everything the agent cannot do: deciding, checking, and understanding.

## How to use this notebook

This notebook is your **worksheet**. The analysis itself goes into separate notebooks that the agent creates in your repository.

Cells labelled **ANSWER** are plain-text cells for your responses. Cells beginning with `!` are **verification cells**; run them with Shift+Enter. Terminal commands are shown in boxes with `$>` standing for the prompt.

If you run short of time, finish Parts 1 to 5 properly before starting Part 6.

Four habits from lecture apply all the way through:

* **Plan before you build.** A wrong plan, implemented perfectly, is still wrong.
* **One verified step, one commit.** A step you cannot check is too big.
* **Paste, don't describe.** Give the agent the real output and the real error.
* **"Done" is a claim, not a result.** Run it yourself and compare.

### Deliverables

Everything is delivered through your repository, `biom5350-dcm-consensus`.

| Part | Deliverable | Where it lives |
|---|---|---|
| 1 | Project instruction file | `AGENTS.md` |
| 2 | Decision log (Q3) and the approved analysis plan | this notebook; `plan/analysis_plan.md` |
| 3 | Study 1 notebook, sample manifest, QC figures, results table | `notebooks/`, `results/` |
| 4 | The Study 1 notebook, commented and restyled, with a passing "results unchanged" check | `notebooks/` |
| 5 | Study 2 notebook, sample manifest, QC figures, results table | `notebooks/`, `results/` |
| 6 | Consensus notebook and consensus gene table | `notebooks/`, `results/tables/` |
| 7 | README with an AI-use statement, the saved agent chat, this completed worksheet, and the repository URL (Q27) | `README.md`, `chat/`, `worksheets/` |

## Part 0 — Before you start

You need the repository you created in Coding with AI I (Part 6). Run the verification cell. You should see your commit from that activity, `origin` pointing at your repository on github.com, and the starting folders.

If you did not finish that part, do it now: create a public repository named `biom5350-dcm-consensus` with a README, clone it into `~/biom5350`, and create the folders `worksheets/`, `chat/`, `plan/`, `notebooks/`, `results/figures/`, `results/tables/` and `data/raw/`, with `data/raw/` listed in `.gitignore`.

In [2]:
!cd ~/biom5350/biom5350-dcm-consensus && git log --oneline | head -n 5 && git remote -v && echo '---' && ls -a && echo '---' && cat .gitignore

e5a58d4 Add Coding with AI I worksheet and chat
a825206 Initial commit
origin	git@github.com:wumianyang233/biom5350-dcm-consensus.git (fetch)
origin	git@github.com:wumianyang233/biom5350-dcm-consensus.git (push)
---
.   .git	README.md  data	      plan     worksheets
..  .gitignore	chat	   notebooks  results
---
data/raw/


Now set up the agent so that its record ends up in your repository:

1. In the CoCalc file browser, go into the `chat/` folder of your repository and create a new chat file named `dcm_agent.chat`.
2. Start an agent session in that chat, and set the agent's **working directory** to the top of your repository (`~/biom5350/biom5350-dcm-consensus`).

Do not send it anything yet.

**Q1.** Which AI product and which model are you using today? Record the name and version as the tool shows them, and today's date. You will need this for your methods statement in Part 7.

## Part 1 — Write the project instruction file

A new agent session starts from zero. It knows nothing about this project unless the text is in front of it. A **project instruction file** is a short file at the top of the repository that the agent reads at the start of every session. Many tools, including the one we use, look for a file named `AGENTS.md`.

Write this file **yourself**, in the CoCalc editor. Keep it to about one screen. Use these headings:

```
# Project
# Data
# Conventions
# Layout
# Boundaries
```

What belongs under each heading:

* **Project** — the scientific question, in one or two sentences.
* **Data** — facts the agent cannot infer and must not guess: organism, tissue, the two accessions, the platform of each, which groups are compared.
* **Conventions** — the language, and how you want code written. For this project: R; tidyverse for data handling, tables and plots wherever practical; every code cell commented so that a person can follow it.
* **Layout** — which folder each kind of file goes in.
* **Boundaries** — what must never happen. For this project, at least: files in `data/raw/` are never modified, and raw data is never committed.

Test each line with one question: **would the agent make a mistake without it?** If not, cut it. Leave out general advice such as "write clean code", and anything the agent can read from the files.

Save the file as `AGENTS.md` at the top of your repository, then run the verification cell.

In [3]:
!cd ~/biom5350/biom5350-dcm-consensus && wc -l AGENTS.md && echo '---' && cat AGENTS.md

50 AGENTS.md
---
\# Project

Find genes consistently differentially expressed in DCM compared with non-failing hearts across GSE5406 and GSE42955.

\# Data

\- Human left ventricular heart tissue.

\- GSE5406: GPL96, 86 DCM and 16 controls.

\- GSE42955: GPL6244, 12 DCM and 5 controls.

\- Exclude ischemic cardiomyopathy samples from both comparisons.

\- Check GEO sample labels and metadata before analysis.

\# Conventions

\- Use R, with tidyverse for data handling and plots where possible.

\- Add short comments explaining each code step.

\- Use limma for differential expression and adjusted p &lt; 0.05.

\- Compare genes across studies using gene symbols and matching expression directions.

\# Layout

\- notebooks/: analysis notebooks

\- plan/: analysis plan

\- results/figures/: QC plots

\- results/tables/: result tables

\- chat/: AI conversations

\- worksheets/: class worksheets

\- data/raw/: downloaded data

\# Boundaries

\- Do not modify files in data/raw/ or commit raw 

**Q2.** Name one line you considered adding and then cut, and say why. Then name the one line you think will matter most, and what you expect the agent would do without it.

Commit the file. In the terminal, execute:

```
$> cd ~/biom5350/biom5350-dcm-consensus
$> git add AGENTS.md
$> git commit -m "Add project instruction file"
```

## Part 2 — Build the analysis plan with the agent

You are asking for a **plan**, not for code. Nothing gets downloaded or analyzed in this part.

### 2a. Send your brief

Take the draft brief from Coding with AI I (Q11) and revise it now that you have an instruction file. Before sending, check that it:

* states the goal and names both accessions
* asks for a plan first, and asks the agent to **interview you** about the details before writing it
* says what the agent may do before you approve the plan (read GEO records and metadata) and what it may not (download expression data, run analyses)
* lists the deliverables, including a written plan saved as `plan/analysis_plan.md`
* tells the agent to read `AGENTS.md`

If your tool has a planning mode, in which the agent proposes but changes nothing until you approve, use it.

Send the brief.

### 2b. Answer the interview and keep a decision log

The agent will ask you questions. Answer them from your own judgment and from your Coding with AI I answers; do not accept a "recommended" option without knowing why it is recommended. If you do not understand a question, **stop and learn**: open a separate chat and ask what the choice means. Then come back.

As you go, record each decision below. For **Raised by**, write `me` if it was in your brief or you brought it up, `agent` if the agent asked, or `nobody` if it never came up and you had to raise it at the end.

Your plan must cover all twelve areas. If the agent has not asked about one by the time it offers to write the plan, raise it yourself.

**Q3. Decision log.**

### 2c. Get the written plan

When the interview is finished, have the agent write the plan to `plan/analysis_plan.md`. Whatever you decided, the plan must be organized so that it produces:

* **one notebook per study** and **one notebook for the consensus**, in `notebooks/`, written in R, each ending with `sessionInfo()`
* for each study, a **sample manifest**: a table listing every sample in the series, whether it is included, its group, and the reason for any exclusion
* for each study, **QC figures** and a **complete results table** (every feature tested, not only the significant ones) with gene identifiers
* a **consensus table**
* a **check for every step**: something you can look at to know the step worked

### 2d. Audit the plan

Read the plan from top to bottom, with your decision log beside it. A plan is far cheaper to fix than code.

In [4]:
!cd ~/biom5350/biom5350-dcm-consensus && wc -l plan/analysis_plan.md && echo '---' && cat plan/analysis_plan.md

66 plan/analysis_plan.md
---
# Analysis plan: DCM versus non-failing hearts

## Objective

Identify genes differentially expressed in dilated cardiomyopathy (DCM) compared with non-failing controls in GSE5406 and GSE42955. Analyze the studies separately because they use different platforms, then identify gene symbols that are significant in both studies and have the same expression direction.

## Studies and sample selection

Use only samples explicitly labeled as DCM or non-failing/normal controls. Exclude ischemic cardiomyopathy (ICM) samples. Verify the sample labels against GEO metadata and match the selected GSM accessions to matrix columns before analysis. Save a sample information table for each study with the GSM, original label, assigned group, inclusion status, and reason for the decision. Do not remove any other sample without checking and documenting why.

The previously reviewed labels indicate the following expected counts; confirm them against the matrices before analysi

**Q4.** Compare the plan with your decision log, area by area. List every place where the plan says something different from what you decided, or leaves a decision out. For each one, say how you had the agent correct it. If you found none, say which three areas you checked most carefully.

**Q5.** List the choices that appear in the plan that you were never asked about. These are decisions the agent made for you. For each, do you accept it, and on what basis?

**Q6.** Pick the one step in the plan you understand least. Open a **separate** chat and ask about it: what the step does to the data, what it assumes, and what a wrong result would look like. Summarize what you learned in two or three sentences, and say how you could confirm it from a source outside the model.

**Q7.** Using your decision log, how many of the twelve areas did the agent raise, how many did you raise, and how many did nobody raise until you checked? What does that tell you about relying on the interview alone?

### 2e. Approve and commit

When the plan matches your decisions, tell the agent the plan is approved, but that it should **wait for your instruction** before starting each part. Then commit the plan:

```
$> git add plan/analysis_plan.md
$> git commit -m "Add approved analysis plan"
```

## Part 3 — Study 1: GSE5406

Start with GSE5406. Ask the agent to carry out the plan for **this study only**, and to **stop after each stage** below and show you its check before going on:

1. download the data and build the sample manifest
2. confirm the scale and produce the QC figures
3. fit the model and write the results table

Before each stage, write your prediction. After it, look at the real output and compare. Use your Coding with AI I answers (Q4, Q5 and Q10) for the predictions.

### 3a. Samples

**Q8. Predict, then check.** Before the agent runs stage 1: how many samples are in the series, how many should be in the analysis in each group, and how many excluded? After it runs: open the sample manifest yourself. Do the counts match? Where did the group label for each sample come from?

### 3b. Scale and quality control

**Q9.** What did the agent do to confirm the values are on a log2 scale, and what is the range? Look at the distribution plot of all samples. Is there anything about it that seems too good, and what in the GEO record (Coding with AI I, Q6) explains it?

**Q10.** Look at the QC figures. Was any sample excluded? If so, does the exclusion follow the rule in your plan? If the agent wants to drop a sample for a reason your rule does not cover, what do you tell it?

### 3c. The model and the results

**Q11. Predict, then check.** Before stage 3: how many features will be tested? Which group is the reference, so what does a positive log2 fold change mean? After it runs: were you right? Find where in the code the reference group is set.

**Q12. Check one number by hand.** Take the top-ranked feature in the results table. In a new cell that **you** add to the study notebook, compute the mean expression in each group for that feature and take the difference. Does it equal the reported log2 fold change, with the sign you expected? Record the three numbers.

**Q13.** How many features are significant at your threshold? How many of those also have an absolute log2 fold change of at least 1? If the two numbers are very different, what does that tell you about this study, and does it change what you want to call "differentially expressed"?

**Q14. Known biology.** Look up the gene or genes you named in Coding with AI I (Q10) in the results table. Do they behave as expected? If a gene you expected is missing or flat, list two possible explanations that are not "the biology is wrong".

### 3d. Run it yourself

The agent has told you what the notebook produced. That report is model output too. In the Study 1 notebook, choose **Kernel ▸ Restart and run all**, and wait for it to finish.

**Q15.** Did the notebook run from top to bottom without the agent? Do the sample counts and the number of significant features match what the agent reported in the chat? If anything differed or failed, paste the actual message to the agent and describe what was wrong.

Commit Study 1: the notebook, the manifest, the figures and the results table. Check `git status` first and confirm that nothing under `data/raw/` is about to be committed.

```
$> git status
$> git add notebooks results
$> git commit -m "Add GSE5406 analysis: DCM vs non-failing"
```

In [5]:
!cd ~/biom5350/biom5350-dcm-consensus && git log --oneline | head -n 5 && echo '--- tracked files under data/raw (should be none):' && git ls-files data/raw | grep -v gitkeep | wc -l && echo '---' && ls notebooks results/figures results/tables

ed3aaf3 Add GSE5406 analysis: DCM vs non-failing
06fda2e Add approved DCM analysis plan
95b8378 Add project instruction file
e5a58d4 Add Coding with AI I worksheet and chat
a825206 Initial commit
--- tracked files under data/raw (should be none):
0
---
notebooks:
GSE5406_analysis.ipynb

results/figures:
GSE5406_MDS.png	 GSE5406_expression_distributions.png
GSE5406_PCA.png	 GSE5406_sample_clustering.png

results/tables:
GSE5406_QC_sample_metrics.csv
GSE5406_limma_gene_symbol_selected.csv
GSE5406_limma_gene_symbol_significant.csv
GSE5406_limma_probe_level_all.csv
GSE5406_limma_probe_level_significant.csv
GSE5406_limma_summary.csv
GSE5406_sample_info.csv
GSE5406_stage1_validation.csv


## Part 4 — Read before you trust

The Study 1 code now exists and runs. That is not the same as you understanding it. This part uses the agent to close that gap, on the notebook you just committed.

### 4a. Explain

Pick the **two** code cells in the Study 1 notebook that you understand least. For each, ask the agent to explain what the cell does to the data and why it is needed. Tell it what you already know, so the explanation starts at your level. Then check the explanation against the code.

**Q16.** For one of the two cells, explain it back in your own words, as you would to a classmate. Then ask the agent what you got wrong or left out, and record its answer. Do you agree with the correction?

### 4b. Audit against your conventions

Your instruction file asked for tidyverse data handling and for commented code. Read through the notebook and check whether you got that.

* Mark any cell that handles tables in base R (for example `merge()`, `order()`, or indexing with `[ , ]`) where a tidyverse equivalent would be clearer. Bioconductor function calls such as `lmFit()` are not what this is about; leave those alone.
* Mark any cell with no comments, and any comment that only restates the function name or that you could not defend.

**Q17.** What did you find? Give the number of cells you marked for restyling and for commenting, and one example of each. Did the agent follow `AGENTS.md`?

### 4c. Restyle, then prove nothing changed

First keep a copy of the current results table, so you have something to compare against. In the terminal (substitute the real file name):

```
$> cd ~/biom5350/biom5350-dcm-consensus/results/tables
$> cp <study1_results>.csv /tmp/study1_before.csv
```

Now ask the agent to revise the cells you marked: rewrite the table handling with tidyverse functions, and add or fix the comments. Tell it that **the results must not change**. Then restart the kernel and run the whole notebook again yourself.

Finally, compare the old and new tables. Add a cell like this at the end of the Study 1 notebook, with the real file name, and run it:

```r
before <- read.csv("/tmp/study1_before.csv")
after  <- read.csv("../results/tables/<study1_results>.csv")
all.equal(before, after)
```

You are aiming for `TRUE`.

**Q18.** Did the comparison return `TRUE`? If not, paste what it reported, say what the restyling had changed, and describe how you resolved it. Which do you conclude: the rewrite was wrong, or the two tables differ only in a way that does not matter? How do you know?

### 4d. Fix the file, not just the notebook

From lecture: *treat the instruction file like code. When the agent gets something wrong, fix the file.* If the agent did not follow your conventions in Study 1, make the relevant lines in `AGENTS.md` more concrete, so that Study 2 comes out right the first time. If it did follow them, leave the file as it is.

**Q19.** Did you change `AGENTS.md`? Paste the lines before and after, or explain why no change was needed.

Commit the revised notebook, and `AGENTS.md` if you changed it:

```
$> git add -A notebooks results AGENTS.md
$> git commit -m "Comment and restyle GSE5406 notebook; results unchanged"
```

## Part 5 — Study 2: GSE42955

Ask the agent to carry out the plan for GSE42955, with the same stops as before: samples, then scale and QC, then the model.

**Q20. Predict.** Before the agent starts, list three things that must be done differently for this study than for GSE5406. Think about the form the data come in, how the sample-to-group table gets built, and the annotation. After the agent finishes, note anything that differed which you did not predict.

**Q21. Samples.** How many raw files were downloaded, how many are in the analysis in each group, and how many excluded? Open the manifest and check three samples against their GEO sample pages yourself. Could the group have been worked out from the file names?

**Q22. Normalization.** Compare the distribution plots before and after normalization. What changed? Then ask the agent what the normalization method assumes about the arrays, and whether that is reasonable for 17 heart samples. Record its answer in one or two sentences, and say whether you find it convincing.

**Q23. Results.** How many features were tested, and how many are significant at the same threshold you used for Study 1? Compare with Study 1 (Q13). Give the main reason the two numbers differ, and say why it is not evidence that DCM affects fewer genes in this study.

**Q24.** Did the agent follow your conventions this time without being reminded? Restart the kernel and run the Study 2 notebook yourself. Do the numbers match what the agent reported?

Commit Study 2:

```
$> git status
$> git add notebooks results
$> git commit -m "Add GSE42955 analysis: DCM vs non-failing"
```

## Part 6 — Combine the studies

You now have two results tables from two different array platforms. Ask the agent to build the consensus notebook according to the rule in your plan (decision areas 7 and 9). Have it stop and report after matching the two tables by gene, before it applies the consensus rule.

**Q25. Matching.** How many genes are in each study's table after mapping to genes, and how many are in both? What happened to the features that did not map to a gene, and to genes with several probe sets? Is that what your plan said?

**Q26. Consensus.** Record: how many genes meet your consensus rule; of the genes significant in both studies, what fraction change in the same direction; and your top ten consensus genes with their direction. If the same-direction fraction were close to one half, what would you suspect had gone wrong?

Restart the kernel and run the consensus notebook yourself, then commit:

```
$> git add notebooks results
$> git commit -m "Add cross-study consensus of DCM vs non-failing"
```

**If you have time:** a strict "significant in both studies" rule depends heavily on the smaller study. Ask the agent to propose one statistical way to combine the two studies that uses the size of the effect and its uncertainty in each, and to explain what it assumes. Add it to the consensus notebook as a clearly labelled extra section, and compare the two gene lists.

## Part 7 — Publish

### 7a. Write the README

Edit `README.md` yourself so that a stranger could understand and rerun the project. Include:

* the question, the two datasets, and the comparison
* how to reproduce it: which notebooks to run, in what order
* the main result: how many consensus genes, and where the table is
* **AI use.** Which tool and model you used and on what date (Q1), what you used it for, and how you verified its output. Be specific: name the checks you actually ran.
* **Limitations and trust.** Three short lists: what you **verified** yourself, what you **tested** by rerunning, and what you **took on faith**.

### 7b. Check the chat record

Your conversation with the agent is in `chat/dcm_agent.chat`. Look through it before you publish. It can contain account identifiers, file paths, and anything you pasted. If your separate "learning" chat from Q6 is useful to keep, copy it into `chat/` too.

### 7c. Add this worksheet, commit and push

Save this notebook and copy it into `worksheets/` in your repository, beside the Coding with AI I worksheet. Then:

```
$> git status
$> git add README.md chat worksheets
$> git commit -m "Add README with AI-use statement, agent chat and worksheet"
$> git push
```

Run the final verification cell, then open your repository on github.com and confirm that the files are there.

In [6]:
!cd ~/biom5350/biom5350-dcm-consensus && git status --short | head -n 10 && echo '--- log:' && git log --oneline && echo '--- tracked files:' && git ls-files && echo '--- raw data tracked (should be 0):' && git ls-files data/raw | grep -v gitkeep | wc -l

?? data/processed/
--- log:
ac2d0ce (HEAD -> main, origin/main, origin/HEAD) Add README with AI-use statement, agent chat and worksheet
62bf2e4 Add cross-study consensus of DCM vs non-failing
af98f37 Add GSE42955 analysis: DCM vs non-failing
9070053 Comment and restyle GSE5406 notebook; results unchanged
ed3aaf3 Add GSE5406 analysis: DCM vs non-failing
06fda2e Add approved DCM analysis plan
95b8378 Add project instruction file
e5a58d4 Add Coding with AI I worksheet and chat
a825206 Initial commit
--- tracked files:
.gitignore
AGENTS.md
README.md
chat/dcm_agent.chat
chat/part5.sage-chat
data/raw/.gitkeep
notebooks/.gitkeep
notebooks/Consensus_analysis.ipynb
notebooks/GSE42955_analysis.ipynb
notebooks/GSE5406_analysis.ipynb
plan/.gitkeep
plan/analysis_plan.md
results/figures/.gitkeep
results/figures/GSE42955_MDS.png
results/figures/GSE42955_PCA.png
results/figures/GSE42955_expression_distributions.png
results/figures/GSE42955_sample_clustering.png
results/figures/GSE5406_MDS.png
results/

**Q27.** Paste the URL of your repository.

## Reflection

Answer these after class if you run out of time.

**Q28.** Describe one thing the agent got wrong, left out, or decided without telling you, and how you caught it. If you caught nothing, describe the check that gave you the most confidence, and one thing you did not check.

**Q29.** From your README's "took on faith" list, choose the item that worries you most. What would it take to move it to the "verified" list?

**Q30.** If a committee member asked you to defend your consensus gene list, which single decision in your decision log would you expect them to challenge first, and what would you say?

## What should be in your repository

When you are finished, the repository at the URL you gave in Q27 should have:

* `AGENTS.md`
* `plan/analysis_plan.md`
* three analysis notebooks in `notebooks/`, each running from top to bottom and ending with `sessionInfo()`
* sample manifests, QC figures, two per-study results tables and a consensus table in `results/`
* `chat/dcm_agent.chat`
* `README.md` with the AI-use statement and the three trust lists
* both completed worksheets in `worksheets/`
* a commit history with one commit per verified step
* nothing from `data/raw/` committed